    email agent
    - authenticates user
        - only then are they allowed into the "inbox"
        - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
    - checks "inbox"
        - email in tool
    - sends emails
        - human in the loop

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [47]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b",
    reasoning=False
)

In [48]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )

In [49]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Please check my inbox")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I cannot check your inbox as I do not have access to your email account or any external services. Please use your email client or webmail to check your inbox. Let me know if there's anything else I can assist you with!


In [50]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

C:\Users\ASUS\miniconda3\envs\Ollama\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


I found an email in your inbox from Jane (jane@example.com) inviting you to grab a coffee next week. Would you like me to help you respond to this email?


In [51]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Can you provide me with the email?")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Sure! Here is the email you received:

---

**From:** Jane (jane@example.com)  
**Message:**  
Hi Julie,  
I'm going to be in town next week and was wondering if we could grab a coffee?  
- best, Jane (jane@example.com)  

---

Would you like to respond to this email?


In [52]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Draft a reply to Jane's email and send it")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Of course! Here's a draft reply to Jane's email:

---

**Subject:** Re: Coffee Meeting

Hi Jane,

That sounds great! I'd love to grab a coffee next week. Could you let me know your preferred time and place?

Looking forward to hearing from you!

Best regards,  
Julie

---

Would you like to make any changes to this draft before I send it?


In [54]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Send it")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [55]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,

That sounds great! I'd love to grab a coffee next week. Could you let me know your preferred time and place?

Looking forward to hearing from you!

Best regards,
Julie


In [56]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Your reply has been successfully sent to Jane at jane@example.com with the subject "Re: Coffee Meeting". Let me know if there's anything else I can assist you with!


In [57]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='Please check my inbox', additional_kwargs={}, response_metadata={}, id='098f6a2a-dce8-45b9-8968-f3b816d2c909'),
              AIMessage(content="I cannot check your inbox as I do not have access to your email account or any external services. Please use your email client or webmail to check your inbox. Let me know if there's anything else I can assist you with!", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-28T15:03:30.3467072Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1246335600, 'load_duration': 30727500, 'prompt_eval_count': 172, 'prompt_eval_duration': 301261100, 'eval_count': 48, 'eval_duration': 911312000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019c0521-864a-7771-8272-3444c60a25ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 48, 'total_tokens': 220}),
              HumanM